In [20]:
import json
from pathlib import Path
import pandas as pd
import numpy as np


def load_data(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return pd.DataFrame(data["sheets"][0]["rows"])


df = load_data(Path(r"D:\data_project\data\Data.json"))
df.head(5)

,مبلغ إجمالي,قيمة التوصيل,الخدمات,التأمينات,الضرائب,الخصم,الاجمالي,عدد الفواتير,عدد الفواتير_2,اسم الفرع,التاريخ
0,5713.0,0,0,0,4967.83,4967.83,4967.83,326,326,فرع اكتوبر,01/01/2026
1,6092.0,0,0,0,5297.39,5297.39,5297.39,290,290,فرع الشيخ زايد,01/01/2026
2,14785.0,0,0,0,12856.52,12856.52,12856.52,339,339,فرع التجمع,01/01/2026
3,3990.0,0,0,0,3469.57,3469.57,3469.57,218,218,فرع اكتوبر,02/01/2026
4,4739.0,0,0,0,4120.87,4120.87,4120.87,231,231,فرع الشيخ زايد,02/01/2026


In [ ]:
is_identical = df["عدد الفواتير"].equals(df["عدد الفواتير_2"])
print(is_identical)  # Returns True or False

In [ ]:
is_single_value = df["قيمة التوصيل"].nunique() == 1
print(is_single_value)  # True or False
is_single_value = df["الخدمات"].nunique() == 1
print(is_single_value)  # True or False
is_single_value = df["التأمينات"].nunique() == 1
print(is_single_value)  # True or False

In [ ]:
is_identical = df["الضرائب"].equals(df["الخصم"])
print(is_identical)  # Returns True or False
is_identical = df["الضرائب"].equals(df["الاجمالي"])
print(is_identical)  # Returns True or False

In [ ]:
df["الاجمالي بعد الخصم"] = df["الاجمالي"]
df = df.drop(columns=["الاجمالي", "الخصم", "الضرائب"])

In [ ]:
df = df.drop(columns=["قيمة التوصيل", "الخدمات", "التأمينات", "عدد الفواتير_2"])

In [ ]:
df.head(5)

In [21]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """Clean sales dataframe by creating final total and removing redundant columns."""
    # Work on a copy to prevent mutating the original DataFrame outside the function
    df_clean = df.copy()

    df_clean["الاجمالي بعد الخصم"] = df_clean["الاجمالي"]

    columns_to_drop = [
        "الاجمالي",
        "الخصم",
        "الضرائب",
        "قيمة التوصيل",
        "الخدمات",
        "التأمينات",
        "عدد الفواتير_2",
    ]

    return df_clean.drop(columns=columns_to_drop, errors="ignore")

In [22]:
df = clean(df)
df.head(5)

,مبلغ إجمالي,عدد الفواتير,اسم الفرع,التاريخ,الاجمالي بعد الخصم
0,5713.0,326,فرع اكتوبر,01/01/2026,4967.83
1,6092.0,290,فرع الشيخ زايد,01/01/2026,5297.39
2,14785.0,339,فرع التجمع,01/01/2026,12856.52
3,3990.0,218,فرع اكتوبر,02/01/2026,3469.57
4,4739.0,231,فرع الشيخ زايد,02/01/2026,4120.87


In [ ]:
cols_to_convert = ["مبلغ إجمالي", "الاجمالي بعد الخصم", "عدد الفواتير"]
df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors="coerce")

In [ ]:
df.head(5)

In [ ]:
# 1. Total sum for each column
total_gross_amount = df["مبلغ إجمالي"].sum()
total_after_discount = df["الاجمالي بعد الخصم"].sum()
total_invoices = df["عدد الفواتير"].sum()

# 2. Overall average cost per invoice
overall_avg_per_invoice = total_gross_amount / total_invoices

print(f"إجمالي المبلغ الإجمالي: {total_gross_amount:,.2f}")
print(f"إجمالي بعد الخصم: {total_after_discount:,.2f}")
print(f"إجمالي عدد الفواتير: {total_invoices:,}")
print(f"متوسط قيمة الفاتورة الكلي: {overall_avg_per_invoice:,.2f}")

In [ ]:
branch_dfs = {}  # Start with an empty dictionary

for branch_name, branch_data in df.groupby("اسم الفرع"):
    branch_dfs[branch_name] = branch_data  # Store DataFrame under its branch name key

In [ ]:
# Get October branch data
october_df = branch_dfs["فرع اكتوبر"]

# Perform standard pandas operations on it
october_df.head()

In [ ]:
for branch_name, branch_df in branch_dfs.items():
    total_sales = branch_df["مبلغ إجمالي"].sum()
    total_after_discount = branch_df["الاجمالي بعد الخصم"].sum()
    total_invoices = branch_df["عدد الفواتير"].sum()

    # Calculate average revenue per invoice (handling division by zero safely)
    overall_avg_per_invoice = (
        total_sales / total_invoices if total_invoices > 0 else 0
    )

    print(f"=== {branch_name} ===")
    print(f"  - إجمالي المبيعات (قبل الخصم): {total_sales:,.2f} EGP")
    print(f"  - إجمالي بعد الخصم: {total_after_discount:,.2f} EGP")
    print(f"  - عدد الفواتير: {total_invoices:,}")
    print(f"  - نسبة النقود لكل فاتورة (المتوسط): {overall_avg_per_invoice:,.2f} EGP\n")

In [ ]:
def get_sales_insights(df: pd.DataFrame) -> pd.DataFrame:
    """Calculates aggregated sales metrics across all branches and appends an overall total row.

    Args:
        df (pd.DataFrame): Cleaned input DataFrame containing branch sales.

    Returns:
        pd.DataFrame: A single summary table with branch-level metrics and a total row.
    """
    # 1. Work on a copy to prevent mutating the input DataFrame
    df_work = df.copy()

    # 2. Guarantee numeric types for key columns
    numeric_cols = ["مبلغ إجمالي", "الاجمالي بعد الخصم", "عدد الفواتير"]
    for col in numeric_cols:
        if col in df_work.columns:
            df_work[col] = pd.to_numeric(df_work[col], errors="coerce").fillna(0)

    # 3. Aggregate totals per branch
    branch_summary = (
        df_work.groupby("اسم الفرع", as_index=False)[numeric_cols]
        .sum()
    )

    # 4. Calculate average cost per invoice per branch
    branch_summary["نسبة النقود لكل فاتورة"] = np.where(
        branch_summary["عدد الفواتير"] > 0,
        (
            branch_summary["مبلغ إجمالي"] / branch_summary["عدد الفواتير"]
        ).round(2),
        0,
    )

    # 5. Compute overall total across all branches
    total_gross = df_work["مبلغ إجمالي"].sum()
    total_net = df_work["الاجمالي بعد الخصم"].sum()
    total_invoices = df_work["عدد الفواتير"].sum()
    overall_avg = (
        round(total_gross / total_invoices, 2) if total_invoices > 0 else 0.0
    )

    overall_row = pd.DataFrame([
        {
            "اسم الفرع": "الإجمالي العام (جميع الفروع)",
            "مبلغ إجمالي": total_gross,
            "الاجمالي بعد الخصم": total_net,
            "عدد الفواتير": total_invoices,
            "نسبة النقود لكل فاتورة": overall_avg,
        }
    ])

    # 6. Return combined table
    return pd.concat([branch_summary, overall_row], ignore_index=True)

In [ ]:
df = get_sales_insights(df)
df.head(5)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Ensure date column is datetime
df["التاريخ"] = pd.to_datetime(df["التاريخ"], format="%d/%m/%Y")

# Create a Year-Month column (e.g., '2026-01')
df["الشهر"] = df["التاريخ"].dt.to_period("M").astype(str)

# Aggregate monthly sales by branch
monthly_sales = (
    df.groupby(["الشهر", "اسم الفرع"])["مبلغ إجمالي"].sum().reset_index()
)

In [ ]:
plt.figure(figsize=(12, 6))

# Plot monthly trends for each branch
sns.lineplot(
    data=monthly_sales,
    x="الشهر",
    y="مبلغ إجمالي",
    hue="اسم الفرع",
    marker="o",
    linewidth=2.5,
)

plt.title("Monthly Sales Trend by Branch", fontsize=14, fontweight="bold")
plt.xlabel("Month", fontsize=12)
plt.ylabel("Total Sales (EGP)", fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(title="Branch", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

plt.show()

In [ ]:
# Ensure numeric type for invoice counts
df["عدد الفواتير"] = pd.to_numeric(df["عدد الفواتير"], errors="coerce").fillna(
    0
)

# Aggregate monthly total invoices by branch
monthly_invoices = (
    df.groupby(["الشهر", "اسم الفرع"])["عدد الفواتير"].sum().reset_index()
)

In [ ]:
# Set figure size and style
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

# Create line plot
ax = sns.lineplot(
    data=monthly_invoices,
    x="الشهر",
    y="عدد الفواتير",
    hue="اسم الفرع",
    marker="o",
    linewidth=2.5,
    markersize=8,
)

# Customize titles and labels
plt.title(
    "Monthly Invoices Volume by Branch (عدد الفواتير)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Total Invoices Count", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Branch", bbox_to_anchor=(1.05, 1), loc="upper left")

# Display value annotations on points
for p in ax.patches:
    pass  # Line plots use markers; add annotations if needed

plt.tight_layout()
plt.show()

In [23]:
def generate_sales_visualizations(
    df: pd.DataFrame, output_dir: str | Path = "charts"
) -> dict[str, Path]:
    """Generates monthly sales and invoice trend visualizations across branches and saves them as high-res PNG images.

    Args:
        df (pd.DataFrame): Cleaned input DataFrame.
        output_dir (str | Path): Directory path where chart images will be saved.

    Returns:
        dict[str, Path]: Dictionary mapping plot names to saved image file paths.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    # 1. Work on a copy to maintain data immutability
    df_work = df.copy()

    # 2. Defensive type conversion and Month creation
    df_work["التاريخ"] = pd.to_datetime(
        df_work["التاريخ"], format="%d/%m/%Y", errors="coerce"
    )
    df_work["الشهر"] = df_work["التاريخ"].dt.to_period("M").astype(str)

    numeric_cols = ["مبلغ إجمالي", "عدد الفواتير"]
    for col in numeric_cols:
        if col in df_work.columns:
            df_work[col] = pd.to_numeric(df_work[col], errors="coerce").fillna(0)

    # 3. Monthly aggregation per branch
    monthly_data = (
        df_work.groupby(["الشهر", "اسم الفرع"])[numeric_cols]
        .sum()
        .reset_index()
    )

    sns.set_theme(style="whitegrid")
    saved_charts = {}

    # --- Chart 1: Monthly Sales Revenue Trend ---
    fig_sales, ax_sales = plt.subplots(figsize=(12, 6))
    sns.lineplot(
        data=monthly_data,
        x="الشهر",
        y="مبلغ إجمالي",
        hue="اسم الفرع",
        marker="o",
        linewidth=2.5,
        markersize=8,
        ax=ax_sales,
    )
    ax_sales.set_title(
        "Monthly Sales Trend by Branch (مبلغ إجمالي)",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    ax_sales.set_xlabel("Month", fontsize=12)
    ax_sales.set_ylabel("Total Gross Sales (EGP)", fontsize=12)
    ax_sales.tick_params(axis="x", rotation=45)
    ax_sales.legend(title="Branch", bbox_to_anchor=(1.05, 1), loc="upper left")
    fig_sales.tight_layout()

    sales_path = output_path / "monthly_sales_trend.png"
    fig_sales.savefig(sales_path, dpi=300)
    plt.close(fig_sales)  # Free memory
    saved_charts["sales_trend"] = sales_path

    # --- Chart 2: Monthly Invoices Volume Trend ---
    fig_inv, ax_inv = plt.subplots(figsize=(12, 6))
    sns.lineplot(
        data=monthly_data,
        x="الشهر",
        y="عدد الفواتير",
        hue="اسم الفرع",
        marker="o",
        linewidth=2.5,
        markersize=8,
        ax=ax_inv,
    )
    ax_inv.set_title(
        "Monthly Invoices Count Trend by Branch (عدد الفواتير)",
        fontsize=14,
        fontweight="bold",
        pad=15,
    )
    ax_inv.set_xlabel("Month", fontsize=12)
    ax_inv.set_ylabel("Total Invoices Count", fontsize=12)
    ax_inv.tick_params(axis="x", rotation=45)
    ax_inv.legend(title="Branch", bbox_to_anchor=(1.05, 1), loc="upper left")
    fig_inv.tight_layout()

    invoices_path = output_path / "monthly_invoices_trend.png"
    fig_inv.savefig(invoices_path, dpi=300)
    plt.close(fig_inv)  # Free memory
    saved_charts["invoices_trend"] = invoices_path

    return saved_charts

In [24]:
saved_images = generate_sales_visualizations(
    df, output_dir="data\charts"
)

<>:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
C:\Users\ahmed\AppData\Local\Temp\ipykernel_2260\2474084618.py:2: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  df, output_dir="data\charts"
